In [1]:
import h5py
p='/home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup_demo_1.hdf5'
with h5py.File(p,'r') as f:
    print('top:', list(f.keys()))
    if 'data' in f:
        print('data children sample:', list(f['data'].keys())[:5])
        for k in list(f['data'].keys())[:3]:
            g=f['data'][k]
            if isinstance(g,h5py.Group):
                print(k, 'keys:', list(g.keys())[:8])

top: ['data']
data children sample: ['demo_1']
demo_1 keys: ['action_dict', 'actions', 'datagen_info', 'obs', 'states']


In [2]:
import h5py
p = "/home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup_demo_1.hdf5"
with h5py.File(p, "r") as f:
    print(f["data"]["demo_1"]["actions"].shape)
    print("action dim =", f["data"]["demo_1"]["actions"].shape[-1])

(281, 24)
action dim = 24


In [1]:
# Cell 1: 基础依赖
import h5py
from pathlib import Path

hdf5_path = Path("/home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup.hdf5")
print("Exists:", hdf5_path.exists(), "| Size (MB):", round(hdf5_path.stat().st_size / (1024**2), 2))

Exists: True | Size (MB): 4856.97


In [4]:
# Cell 2: 递归收集 HDF5 结构并保存到 txt（group / dataset / attrs）
def collect_h5_structure(h5_obj, lines, indent=0):
    prefix = "  " * indent

    if isinstance(h5_obj, h5py.Group):
        lines.append(f"{prefix}[Group] {h5_obj.name}")
        # 记录 group 属性
        for k, v in h5_obj.attrs.items():
            lines.append(f"{prefix}  - attr: {k} = {v}")
        # 遍历子节点
        for key in h5_obj.keys():
            collect_h5_structure(h5_obj[key], lines, indent + 1)

    elif isinstance(h5_obj, h5py.Dataset):
        lines.append(f"{prefix}[Dataset] {h5_obj.name} | shape={h5_obj.shape} | dtype={h5_obj.dtype}")
        # 记录 dataset 属性
        for k, v in h5_obj.attrs.items():
            lines.append(f"{prefix}  - attr: {k} = {v}")


output_txt_path = hdf5_path.with_suffix(".structure.txt")
lines = []

with h5py.File(hdf5_path, "r") as f:
    collect_h5_structure(f, lines)

output_txt_path.write_text("\n".join(lines), encoding="utf-8")
print(f"HDF5 结构已保存到: {output_txt_path}")

HDF5 结构已保存到: /home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup.structure.txt


In [3]:
# Cell 3: 只看顶层 key（快速浏览）
with h5py.File(hdf5_path, "r") as f:
    print("Top-level keys:")
    for k in f.keys():
        obj = f[k]
        t = "Group" if isinstance(obj, h5py.Group) else "Dataset"
        print(f" - {k} ({t})")

Top-level keys:
 - data (Group)


In [1]:
import h5py
import numpy as np
from pathlib import Path

src_path = Path("/home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup.hdf5")
dst_path = Path("/home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup_part1.hdf5")
target_len = 272

def copy_group_recursive(src_grp, dst_grp):
    """递归复制 group 的 dataset / subgroup / attrs。"""
    # copy attrs
    for k, v in src_grp.attrs.items():
        dst_grp.attrs[k] = v

    for key, item in src_grp.items():
        if isinstance(item, h5py.Dataset):
            ds = dst_grp.create_dataset(
                key,
                data=item[()],
                compression=item.compression,
                compression_opts=item.compression_opts,
                shuffle=item.shuffle,
                fletcher32=item.fletcher32,
                chunks=item.chunks,
            )
            for ak, av in item.attrs.items():
                ds.attrs[ak] = av
        elif isinstance(item, h5py.Group):
            sub_dst = dst_grp.create_group(key)
            copy_group_recursive(item, sub_dst)

with h5py.File(src_path, "r") as fin, h5py.File(dst_path, "w") as fout:
    # 复制 root attrs
    for k, v in fin.attrs.items():
        fout.attrs[k] = v

    if "data" not in fin:
        raise KeyError("输入文件中不存在 'data' 组")

    src_data = fin["data"]
    dst_data = fout.create_group("data")

    # 先复制 data attrs（后面会更新 total）
    for k, v in src_data.attrs.items():
        dst_data.attrs[k] = v

    demo_keys = sorted([k for k in src_data.keys() if k.startswith("demo_")])
    selected_keys = []

    # 筛选 actions 长度 == 272
    for demo_key in demo_keys:
        demo_grp = src_data[demo_key]
        if "actions" not in demo_grp:
            continue
        action_len = demo_grp["actions"].shape[0]
        if action_len == target_len:
            selected_keys.append(demo_key)

    # 拷贝筛选出来的 demo，并重命名为连续 demo_0, demo_1, ...
    total_samples = 0
    for new_idx, old_key in enumerate(selected_keys):
        old_demo = src_data[old_key]
        new_key = f"demo_{new_idx}"
        new_demo = dst_data.create_group(new_key)

        copy_group_recursive(old_demo, new_demo)

        # 记录来源 demo 名称，便于追踪
        new_demo.attrs["source_demo_key"] = old_key

        # 统计 total（优先用 num_samples，否则用 actions 长度）
        if "num_samples" in old_demo.attrs:
            n = int(old_demo.attrs["num_samples"])
        else:
            n = int(old_demo["actions"].shape[0])
            new_demo.attrs["num_samples"] = n
        total_samples += n

    # 更新 data-level total
    dst_data.attrs["total"] = total_samples

print(f"输入文件: {src_path}")
print(f"输出文件: {dst_path}")
print(f"筛选出的 demo 数量: {len(selected_keys)}")
print(f"输出 data.attrs['total']: {total_samples}")
print("前几个来源 demo:", selected_keys[:10])

输入文件: /home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup.hdf5
输出文件: /home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup_part1.hdf5
筛选出的 demo 数量: 209
输出 data.attrs['total']: 56848
前几个来源 demo: ['demo_1002', 'demo_1004', 'demo_1011', 'demo_1012', 'demo_1017', 'demo_1019', 'demo_1020', 'demo_1022', 'demo_1025', 'demo_105']


In [ ]:
import h5py
from pathlib import Path

# 输入 / 输出文件路径
src_path = Path("/home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup_pointviewpc.hdf5")
dst_path = Path("/home/benhua/DexSim/dexmimicgen/datasets/generated/two_arm_drawer_cleanup_pointviewpc_only_valid.hdf5")
required_key = "pointview_pc"


def copy_group_recursive(src_grp, dst_grp):
    """递归复制 group 的 dataset / subgroup / attrs。"""
    for k, v in src_grp.attrs.items():
        dst_grp.attrs[k] = v

    for key, item in src_grp.items():
        if isinstance(item, h5py.Dataset):
            ds = dst_grp.create_dataset(
                key,
                data=item[()],
                compression=item.compression,
                compression_opts=item.compression_opts,
                shuffle=item.shuffle,
                fletcher32=item.fletcher32,
                chunks=item.chunks,
            )
            for ak, av in item.attrs.items():
                ds.attrs[ak] = av
        elif isinstance(item, h5py.Group):
            sub_dst = dst_grp.create_group(key)
            copy_group_recursive(item, sub_dst)


with h5py.File(src_path, "r") as fin, h5py.File(dst_path, "w") as fout:
    # root attrs
    for k, v in fin.attrs.items():
        fout.attrs[k] = v

    if "data" not in fin:
        raise KeyError("输入文件中不存在 'data' 组")

    src_data = fin["data"]
    dst_data = fout.create_group("data")

    # data attrs
    for k, v in src_data.attrs.items():
        dst_data.attrs[k] = v

    demo_keys = sorted(src_data.keys(), key=lambda x: (0, int(x[5:])) if x.startswith("demo_") else (1, str(x)))
    kept = []
    dropped = []
    total_samples = 0

    for old_key in demo_keys:
        old_demo = src_data[old_key]
        obs_grp = old_demo.get("obs", None)
        if (obs_grp is None) or (required_key not in obs_grp):
            dropped.append(old_key)
            continue

        # 保持原 demo 名，不重命名
        new_demo = dst_data.create_group(old_key)
        copy_group_recursive(old_demo, new_demo)
        kept.append(old_key)

        if "num_samples" in old_demo.attrs:
            n = int(old_demo.attrs["num_samples"])
        else:
            n = int(old_demo["actions"].shape[0]) if "actions" in old_demo else 0
            new_demo.attrs["num_samples"] = n
        total_samples += n

    dst_data.attrs["total"] = total_samples

    # 记录过滤统计，便于追踪
    fout.attrs["filter_rule"] = f"keep demos with obs/{required_key}"
    fout.attrs["kept_demo_count"] = len(kept)
    fout.attrs["dropped_demo_count"] = len(dropped)

print(f"输入文件: {src_path}")
print(f"输出文件: {dst_path}")
print(f"保留 demo 数量: {len(kept)}")
print(f"删除 demo 数量: {len(dropped)}")
print(f"data.attrs['total']: {total_samples}")
print("前 10 个删除的 demo:", dropped[:10])